In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [122]:
df = pd.read_csv('test_synthetic_15000.csv', delimiter=';', encoding='utf-8')
# encoding='utf-8' is used to read the file with the correct encoding, especially if it contains special characters.


In [123]:
display(df.describe())
display(df.info())
display(df.head(5))

,Sequence
count,15000.000000
mean,147445.475333
std,11966.584991
min,126945.000000
25%,137081.000000
50%,147394.500000
75%,157824.250000
max,168178.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Sequence       15000 non-null  int64 
 1   Mission Name   15000 non-null  object
 2   Robot Name     15000 non-null  object
 3   Map Id         15000 non-null  object
 4   Robot Id       15000 non-null  object
 5   Source         15000 non-null  object
 6   State          15000 non-null  object
 7   State Outcome  15000 non-null  object
 8   Created At     15000 non-null  object
 9   Started At     15000 non-null  object
 10  Completed At   15000 non-null  object
dtypes: int64(1), object(10)
memory usage: 1.3+ MB


None

,Sequence,Mission Name,Robot Name,Map Id,Robot Id,Source,State,State Outcome,Created At,Started At,Completed At
0,126945,[Z3L Tek Modbus] DoluPalet --> ÇelikPalet,SEIT1500F - 5,68da91613ea5ec945691c1fd,69383c614a79835bdcf8c9a5,api,finished,Finished successfully,02/27/2026 00:00:53,02/27/2026 00:12:40,02/27/2026 00:17:16
1,126950,*[Z1L] DoluPalet --> Streç (tek modbus),SEIT1500F - 6,68da91613ea5ec945691c1fd,695dfe67ee2bb67f52f928d0,api,finished,Finished successfully,02/27/2026 00:05:13,02/27/2026 00:18:38,02/27/2026 00:26:24
2,126951,[Z2L Tek Modbus] DoluPalet --> Streç,SEIT1500F - 5,68da91613ea5ec945691c1fd,69383c614a79835bdcf8c9a5,api,finished,Finished successfully,02/27/2026 00:07:11,02/27/2026 00:22:12,02/27/2026 00:30:59
3,126952,[Z3R Tek Modbus] DoluPalet --> YarımPalet,SEIT1500F - 2,68da91613ea5ec945691c1fd,69085a563ccfb44a43d8ebc1,api,finished,Finished successfully,02/27/2026 00:08:08,02/27/2026 00:26:07,02/27/2026 00:28:51
4,126956,*[Z1R] DoluPalet --> Kolon,SEIT1500F - 6,68da91613ea5ec945691c1fd,695dfe67ee2bb67f52f928d0,api,finished,Finished successfully,02/27/2026 00:20:29,02/27/2026 00:27:18,02/27/2026 00:31:23


In [124]:
# Split the date columns into "%m/%d/%Y" and "%H:%M:%S"
objects = ['Created At', 'Started At', 'Completed At']
for obj in objects:
    df[obj] = pd.to_datetime(df[obj], format='%m/%d/%Y %H:%M:%S')
    
df[objects].dtypes

# Extract
df['Queue Time'] = df['Started At'] - df['Created At']
df['Execution Time'] = df['Completed At'] - df['Started At']
display(df[['Queue Time', 'Execution Time']].head(5))

# Converting to seconds 
df['Queue Time'] = df['Queue Time'].dt.total_seconds()
df['Execution Time'] = df['Execution Time'].dt.total_seconds()
display(df[['Queue Time', 'Execution Time']].head(5))


,Queue Time,Execution Time
0,0 days 00:11:47,0 days 00:04:36
1,0 days 00:13:25,0 days 00:07:46
2,0 days 00:15:01,0 days 00:08:47
3,0 days 00:17:59,0 days 00:02:44
4,0 days 00:06:49,0 days 00:04:05


,Queue Time,Execution Time
0,707.0,276.0
1,805.0,466.0
2,901.0,527.0
3,1079.0,164.0
4,409.0,245.0


In [125]:
# Z1L, Z1R, Z2L gibi görev kodunu çıkar
df["Mission Zone"] = df["Mission Name"].str.extract(r"(Z\d+[LR])", expand=False)


def extract_task_type(name):
    name_lower = str(name).lower()
    
    if "stre" in name_lower:
        return "Streç"
    elif "kolon" in name_lower:
        return "Kolon"
    elif "çelik" in name_lower or "celik" in name_lower:
        return "ÇelikPalet"
    elif "yar" in name_lower:
        return "YarımPalet"
    else:
        return "Other"


df["Task Type"] = df["Mission Name"].apply(extract_task_type)
df = df[~df["Task Type"].str.contains("yar|other", case=False, na=False)].copy()

df["Mission Group"] = df["Mission Zone"] + " " + df["Task Type"] 
df["Mission Group"].value_counts()
df['Task Type'].value_counts()

Task Type
Streç         9344
ÇelikPalet    1859
Kolon          809
Name: count, dtype: int64

In [126]:
# Matching Robot Name and Robot Id columns
df = df.rename(columns={'Robot Name': 'Robot#'})
# Split the every values in column called Robot# with "- " and take [1]
df['Robot#'] = df['Robot#'].str.split('- ').str[1]
df.drop('Robot Id', inplace=True, axis=1)
display(df.head(5))

,Sequence,Mission Name,Robot#,Map Id,Source,State,State Outcome,Created At,Started At,Completed At,Queue Time,Execution Time,Mission Zone,Task Type,Mission Group
0,126945,[Z3L Tek Modbus] DoluPalet --> ÇelikPalet,5,68da91613ea5ec945691c1fd,api,finished,Finished successfully,2026-02-27 00:00:53,2026-02-27 00:12:40,2026-02-27 00:17:16,707.0,276.0,Z3L,ÇelikPalet,Z3L ÇelikPalet
1,126950,*[Z1L] DoluPalet --> Streç (tek modbus),6,68da91613ea5ec945691c1fd,api,finished,Finished successfully,2026-02-27 00:05:13,2026-02-27 00:18:38,2026-02-27 00:26:24,805.0,466.0,Z1L,Streç,Z1L Streç
2,126951,[Z2L Tek Modbus] DoluPalet --> Streç,5,68da91613ea5ec945691c1fd,api,finished,Finished successfully,2026-02-27 00:07:11,2026-02-27 00:22:12,2026-02-27 00:30:59,901.0,527.0,Z2L,Streç,Z2L Streç
4,126956,*[Z1R] DoluPalet --> Kolon,6,68da91613ea5ec945691c1fd,api,finished,Finished successfully,2026-02-27 00:20:29,2026-02-27 00:27:18,2026-02-27 00:31:23,409.0,245.0,Z1R,Kolon,Z1R Kolon
5,126957,*[Z2R] DoluPalet --> Kolon,2,68da91613ea5ec945691c1fd,api,finished,Finished successfully,2026-02-27 00:20:54,2026-02-27 00:29:18,2026-02-27 00:33:18,504.0,240.0,Z2R,Kolon,Z2R Kolon


In [127]:
# Drop Unneccesary columns 

df = df.drop(columns=['Map Id','Source','State','State Outcome', 'Mission Zone','Sequence'])
display(df.head(5))

,Mission Name,Robot#,Created At,Started At,Completed At,Queue Time,Execution Time,Task Type,Mission Group
0,[Z3L Tek Modbus] DoluPalet --> ÇelikPalet,5,2026-02-27 00:00:53,2026-02-27 00:12:40,2026-02-27 00:17:16,707.0,276.0,ÇelikPalet,Z3L ÇelikPalet
1,*[Z1L] DoluPalet --> Streç (tek modbus),6,2026-02-27 00:05:13,2026-02-27 00:18:38,2026-02-27 00:26:24,805.0,466.0,Streç,Z1L Streç
2,[Z2L Tek Modbus] DoluPalet --> Streç,5,2026-02-27 00:07:11,2026-02-27 00:22:12,2026-02-27 00:30:59,901.0,527.0,Streç,Z2L Streç
4,*[Z1R] DoluPalet --> Kolon,6,2026-02-27 00:20:29,2026-02-27 00:27:18,2026-02-27 00:31:23,409.0,245.0,Kolon,Z1R Kolon
5,*[Z2R] DoluPalet --> Kolon,2,2026-02-27 00:20:54,2026-02-27 00:29:18,2026-02-27 00:33:18,504.0,240.0,Kolon,Z2R Kolon


In [133]:

display(df.head(5))

,Robot#,Created At,Started At,Completed At,queue_time_seconds,execution_time_seconds,Mission Group
0,5,2026-02-27 00:00:53,2026-02-27 00:12:40,2026-02-27 00:17:16,707.0,276.0,Z3L ÇelikPalet
1,6,2026-02-27 00:05:13,2026-02-27 00:18:38,2026-02-27 00:26:24,805.0,466.0,Z1L Streç
2,5,2026-02-27 00:07:11,2026-02-27 00:22:12,2026-02-27 00:30:59,901.0,527.0,Z2L Streç
4,6,2026-02-27 00:20:29,2026-02-27 00:27:18,2026-02-27 00:31:23,409.0,245.0,Z1R Kolon
5,2,2026-02-27 00:20:54,2026-02-27 00:29:18,2026-02-27 00:33:18,504.0,240.0,Z2R Kolon


In [135]:
df = df.rename(columns={
    "Queue Time": "queue_time_seconds",
    "Execution Time": "execution_time_seconds"
})

# Created , Started and Completed Hour 
df['created_at_hour'] = df['Created At'].dt.hour
df['created_at_minute'] = df['Created At'].dt.minute
df['created_at_month'] = df['Created At'].dt.month
df['created_at_dayofweek'] = df['Created At'].dt.dayofweek



In [137]:
display(df.head(5))
display(df.info())
display(df.describe())

,Robot#,Created At,Started At,Completed At,queue_time_seconds,execution_time_seconds,Mission Group,created_at_hour,created_at_minute,created_at_month,created_at_dayofweek
0,5,2026-02-27 00:00:53,2026-02-27 00:12:40,2026-02-27 00:17:16,707.0,276.0,Z3L ÇelikPalet,0,0,2,4
1,6,2026-02-27 00:05:13,2026-02-27 00:18:38,2026-02-27 00:26:24,805.0,466.0,Z1L Streç,0,5,2,4
2,5,2026-02-27 00:07:11,2026-02-27 00:22:12,2026-02-27 00:30:59,901.0,527.0,Z2L Streç,0,7,2,4
4,6,2026-02-27 00:20:29,2026-02-27 00:27:18,2026-02-27 00:31:23,409.0,245.0,Z1R Kolon,0,20,2,4
5,2,2026-02-27 00:20:54,2026-02-27 00:29:18,2026-02-27 00:33:18,504.0,240.0,Z2R Kolon,0,20,2,4


<class 'pandas.core.frame.DataFrame'>
Index: 12012 entries, 0 to 14999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Robot#                  12012 non-null  object        
 1   Created At              12012 non-null  datetime64[ns]
 2   Started At              12012 non-null  datetime64[ns]
 3   Completed At            12012 non-null  datetime64[ns]
 4   queue_time_seconds      12012 non-null  float64       
 5   execution_time_seconds  12012 non-null  float64       
 6   Mission Group           12012 non-null  object        
 7   created_at_hour         12012 non-null  int32         
 8   created_at_minute       12012 non-null  int32         
 9   created_at_month        12012 non-null  int32         
 10  created_at_dayofweek    12012 non-null  int32         
dtypes: datetime64[ns](3), float64(2), int32(4), object(2)
memory usage: 938.4+ KB


None

,Created At,Started At,Completed At,queue_time_seconds,execution_time_seconds,created_at_hour,created_at_minute,created_at_month,created_at_dayofweek
count,12012,12012,12012,12012.000000,12012.000000,12012.000000,12012.000000,12012.000000,12012.000000
mean,2026-03-07 10:31:38.194472192,2026-03-07 10:34:46.794621952,2026-03-07 10:40:33.369713408,188.600150,346.575092,11.482517,29.707126,2.882118,3.322677
min,2026-02-27 00:00:53,2026-02-27 00:12:40,2026-02-27 00:17:16,0.000000,1.000000,0.000000,0.000000,2.000000,0.000000
25%,2026-03-03 05:41:29.750000128,2026-03-03 05:46:09.249999872,2026-03-03 05:52:12.750000128,15.000000,202.000000,6.000000,15.000000,3.000000,2.000000
50%,2026-03-07 09:15:14,2026-03-07 09:16:37.500000,2026-03-07 09:22:00,93.000000,269.000000,11.000000,30.000000,3.000000,4.000000
75%,2026-03-11 15:26:43.500000,2026-03-11 15:26:44.500000,2026-03-11 15:30:35.750000128,259.000000,384.000000,17.000000,45.000000,3.000000,5.000000
max,2026-03-15 23:46:03,2026-03-15 23:49:43,2026-03-16 00:09:29,2020.000000,2214.000000,23.000000,59.000000,3.000000,6.000000
std,NaN,NaN,NaN,252.937136,260.227035,6.885987,17.482400,0.322482,1.994232


### Training Model - Queue Time 

In [147]:
# Model 1 Queue Time Prediction

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
features = ['Mission Group', 'Robot#', 'created_at_hour', 'created_at_minute', 'created_at_month', 'created_at_dayofweek']
numerical_columns = ['created_at_hour', 'created_at_minute', 'created_at_month', 'created_at_dayofweek']
categorical_columns = ['Mission Group', 'Robot#']

#Standard Scaler
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ]
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

X = df[features]
y = df['queue_time_seconds']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
results = {}

for name, model in models.items():
    pipeline = Pipeline(steps = [
        ('preprocessor' , preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = (y_test, y_pred)
for name, (y_test, y_pred) in results.items():
    print(f"Model: {name}")
    print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f} seconds")
    print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.2f} seconds^2")
    print(f"R^2 Score: {r2_score(y_test, y_pred):.4f}")
    print("-" * 30)


Model: Linear Regression
Mean Absolute Error: 177.80 seconds
Mean Squared Error: 63054.07 seconds^2
R^2 Score: -0.0048
------------------------------
Model: Random Forest
Mean Absolute Error: 189.50 seconds
Mean Squared Error: 70087.63 seconds^2
R^2 Score: -0.1169
------------------------------
Model: Gradient Boosting
Mean Absolute Error: 177.80 seconds
Mean Squared Error: 63831.92 seconds^2
R^2 Score: -0.0172
------------------------------
